# CSP + thermal energy storage

A standalone demonstration with thermal CSP charging, thermal TES state of charge, electric delivery, and integrated CSP+TES economics.

In [ ]:
from pathlib import Path
import sys
import pandas as pd

root = Path.cwd()
while not (root / 'enliten').is_dir():
    if root.parent == root: raise RuntimeError('Run from inside the ENLITEN repository.')
    root = root.parent
if str(root) not in sys.path: sys.path.insert(0, str(root))
from enliten import ChargingPath, Generation, LCOECalculator, Site, Storage, System
data_dir = root / 'examples' / 'data'

def profile(filename):
    frame = pd.read_csv(data_dir / filename)
    return pd.Series(frame['PNM'].to_numpy(float), index=pd.to_datetime(frame.iloc[:, 0], utc=True))

demand_full, csp_full = profile('PNM_demand.csv'), profile('PNM_csp_th_av.csv')
assert demand_full.index.equals(csp_full.index)
hours, start = 24 * 14, pd.Timestamp('2023-07-01', tz='UTC')
window = demand_full.index.get_loc(start)
load = (demand_full.iloc[window:window + hours] * 0.05).rename('load_MW')
tes_capacity_MWh_th, tes_power_MW_e = 2_500.0, 150.0
csp_tes_capex = tes_power_MW_e * 1_000 * 7_912

site = Site('microgrid')
csp = Generation('csp', site, csp_full.iloc[window:window + hours], 'thermal', False, False,
                 capex=csp_tes_capex, opex=tes_power_MW_e * 1_000 * 74.6)
tes = Storage('tes', site, tes_capacity_MWh_th, tes_power_MW_e, 'thermal', 'electric', 0.50,
              maximum_stored_energy_rate_MW=300.0, variable_opex_USD_per_MWh=3.8)
path = ChargingPath('csp', 'tes', 'thermal', 'thermal', 0.90, 300.0 / 0.90)
system = System(load, [tes, csp], [path])
system.timeseries.filter(regex='csp_to_tes|tes_MWh|tes_to_load').head()

In [ ]:
system.operation_metrics()

In [ ]:
LCOECalculator.from_system(system).calculate_lcoe_metrics()

In [ ]:
fig, ax = system.timeseries_plot_source(start_date=0, days=2)
fig

In [ ]:
fig, ax = system.plot_storage_capacity(start_date=0, days=2)
fig

In [ ]:
system.resilience_cases(critical_load_MW=20.0, target_hours=24, n_starts=20, seed=7)
system.resilience_summary